### 연습 문제
- data 폴더 안에 가진 폴더의 모든 json 파일을 하니의 데이터프레임으로 단순 행 결합
- 데이터프레임의 정보를 확인
    - 컬럼들의 타입
    - 결측치 확인
- 결측치가 포함되어 있는 데이터를 추출하여 따로 저장 (df_na)
- 원본 데이터에서는 결측치를 제외
- 'RawText', 'GeneralPolarity'를 제외하고 나머지는 제거
- RawText 컬럼은 텍스트 정규화 (특수문자 제거, 2칸 이사의 공백 처리, 문자열 앞뒤 공백 제거)
    - 글자 수가 1 이하인 데이터를 제외 
- Generality의 데이터는 -1(부정), 0(중립), 1(긍정)
    - 선형 모델에서 차원의 수를 이용하여 높은 위치가 분류 값으로 사용이 되기 때문에 부정을 0으로, 중립을 1로, 긍정을 2로 변환
    - 해당 컬럼의 타입을 int로 변경
- 데이터를 상위 1000개만 추출
- train, test로 데이터를 분할
    - 비율은 5:5
    - 계층화 작업
- 모델은 beomi/kcbert-base 사용
- AutoTokenizer를 이용하여 토큰화 함수 로드 (max_length=128)
- 같은 모델을 로드하여 BertModel + Linear 모델을 정의
- 평가지표 함수를 생성
    - F1-score는 average='macro' 사용
- Trainer, TrainingArguments를 이용하여 학습 관리 객체 생성
- train() 함수를 이용하여 fine_tunning (3회)
- 검증 작업

In [1]:
import re       #텍스트 정규화
#특정 목록의 파일의 목록을 불러오기 위한 라이브러리
import os
from glob import glob

import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME='beomi/kcbert-base'

In [3]:
#특정 경로의 파일의 목록을 불러오기
file_path='../data/가전/'
file_list=os.listdir(file_path)
pd.read_json(file_path + file_list[0])

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."
1,114572,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,SNS,가전,영상/음향가전,(1등급)삼성 QLED 4K TV 138cm(55형) KQ55QT67AFXKR+삼성...,1,317,70,20221119,-1.0,"[{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기..."
2,114573,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,SNS,가전,영상/음향가전,4K HDMI 2.0 양방향 선택기,1,311,70,20221113,0.0,"[{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건..."
3,114574,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,306,84,20221121,-1.0,"[{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ..."
4,114575,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,303,73,20221121,-1.0,"[{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는..."
...,...,...,...,...,...,...,...,...,...,...,...,...
96,114667,이전부터 구매하고 싶어서 계속 눈여겨보다 구매한 락클래식이에요. 포장을 제거하고 제...,SNS,가전,영상/음향가전,엠지텍 락클래식Q9900 (정품),1,290,61,20221120,-1.0,"[{'Aspect': '품질', 'SentimentText': '마감은 좀 문제가 ..."
97,114668,요즘 집에서 작업하면서 핸드폰으로 음악을 들으니 전화를 하거나 핸드폰을 이용할때 자...,SNS,가전,영상/음향가전,오아 아이브릭 휴대용 블루투스 미니 스피커,1,338,78,20221110,-1.0,"[{'Aspect': '디자인', 'SentimentText': '디자인이 좀 그렇..."
98,114669,"처음 들어보는 생소한 브랜드의 tv라 걱정하면서 구입했는데, 역시나 후회 중입니다....",SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,333,79,20221116,-1.0,"[{'Aspect': '소음', 'SentimentText': '별로 소리를 키우지..."
99,114670,최신 기종이라고 해서 기대했는데 구기종보다 못하네요. 소재가 별로여서 예쁜 디자인이...,SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,324,74,20221116,-1.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '최신 기종..."


In [4]:
file_list2=glob('../data/가전/*.json')
pd.read_json(file_list2[0])

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
0,112038,엄마가 갑자기 전화하시더니 집에서 사용하는 노래방 마이크가 사고 싶다고 하시네요.....,SNS,가전,영상/음향가전,(1+1세트) TJ 태진 블루투스 마이크 / 무선 노래방 마이크,3,323,73,20221110,0.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ..."
1,114572,누가 사용하다가 반품한 것만 같은 제품이 와서 조금 당황스러웠어요ㅠㅠ일단 겉 부분에...,SNS,가전,영상/음향가전,(1등급)삼성 QLED 4K TV 138cm(55형) KQ55QT67AFXKR+삼성...,1,317,70,20221119,-1.0,"[{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기..."
2,114573,노트북과 TV를 연결해서 모니터 하나에서 다 보려고 구매했어요.여러 제품과 비교하고...,SNS,가전,영상/음향가전,4K HDMI 2.0 양방향 선택기,1,311,70,20221113,0.0,"[{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건..."
3,114574,너무 별로예요 ㅡㅡ… 이 정도 퀄리티인 줄 알았으면 안 샀을 거 같네요… 음질이 거...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,306,84,20221121,-1.0,"[{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ..."
4,114575,소음이 섞여서 나는 편이에요... 사이즈도 작고 휴대하기 간편해서 손이 자주 가긴 ...,SNS,가전,영상/음향가전,ASMR 방송용 녹음용 유튜버 핀마이크 삼성/LG스마트폰 C타입 스마트폰 M31-C,1,303,73,20221121,-1.0,"[{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는..."
...,...,...,...,...,...,...,...,...,...,...,...,...
96,114667,이전부터 구매하고 싶어서 계속 눈여겨보다 구매한 락클래식이에요. 포장을 제거하고 제...,SNS,가전,영상/음향가전,엠지텍 락클래식Q9900 (정품),1,290,61,20221120,-1.0,"[{'Aspect': '품질', 'SentimentText': '마감은 좀 문제가 ..."
97,114668,요즘 집에서 작업하면서 핸드폰으로 음악을 들으니 전화를 하거나 핸드폰을 이용할때 자...,SNS,가전,영상/음향가전,오아 아이브릭 휴대용 블루투스 미니 스피커,1,338,78,20221110,-1.0,"[{'Aspect': '디자인', 'SentimentText': '디자인이 좀 그렇..."
98,114669,"처음 들어보는 생소한 브랜드의 tv라 걱정하면서 구입했는데, 역시나 후회 중입니다....",SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,333,79,20221116,-1.0,"[{'Aspect': '소음', 'SentimentText': '별로 소리를 키우지..."
99,114670,최신 기종이라고 해서 기대했는데 구기종보다 못하네요. 소재가 별로여서 예쁜 디자인이...,SNS,가전,영상/음향가전,유맥스 139cm 무결점 UHD ／ UHD55L [스탠드형 자가 설치],1,324,74,20221116,-1.0,"[{'Aspect': '제조일/제조사', 'SentimentText': '최신 기종..."


In [5]:
#특정 폴더의 데이터들을 DataFrame으로 생성하여 하나의 데이터프레임으로 결합
df=pd.DataFrame()
for file_name in file_list2:
    data=pd.read_json(file_name)
    #단순 결합
    df=pd.concat([df,data], axis=0)
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [6]:
pd.concat(
    [pd.read_json(file_name) for file_name in file_list2]
).info()

<class 'pandas.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [7]:
pd.concat(
    list(
        map(
            lambda x : pd.read_json(x),
            file_list2
        )
    )
).info()

<class 'pandas.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   str    
 2   Source           4056 non-null   str    
 3   Domain           4056 non-null   str    
 4   MainCategory     4056 non-null   str    
 5   ProductName      4056 non-null   str    
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 3.9+ MB


In [8]:
#결측치 개수 확인
df.isna().sum()

Index                0
RawText              0
Source               0
Domain               0
MainCategory         0
ProductName          0
ReviewScore          0
Syllable             0
Word                 0
RDate                0
GeneralPolarity    378
Aspects              0
dtype: int64

In [9]:
#Polarity 컬럼의 결측치는 따로 추출하여 예측에서 사용할 데이터로 저장
df_na=df.loc[
	df['GeneralPolarity'].isna(),
]
df_na.head()

,Index,RawText,Source,Domain,MainCategory,ProductName,ReviewScore,Syllable,Word,RDate,GeneralPolarity,Aspects
13,114584,귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸...,SNS,가전,영상/음향가전,LG 톤플러스 HBS-830 블루투스 이어폰,1,366,89,20221117,NaN,"[{'Aspect': '소재', 'SentimentText': '귀에서 자꾸 빠져요..."
43,114614,아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지...,SNS,가전,영상/음향가전,삼성 UHD TV (55형)(사은품:삼성 사운드바),1,334,79,20221113,NaN,"[{'Aspect': '품질', 'SentimentText': ' 마감 퀄리티가 별..."
44,114615,화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았...,SNS,가전,영상/음향가전,삼성 UHD TV (55형)(사은품:삼성 사운드바),1,344,85,20221113,NaN,"[{'Aspect': '화질', 'SentimentText': '갑자기 화면에 노이..."
55,114626,"이번에 이사하면서 우리 따님께서 방에 TV가 있으면 좋겠다고 하여, 방에서 사용할 ...",SNS,가전,영상/음향가전,삼성 UHD TV 123cm(49형)(사은품:삼성 사운드바),1,313,73,20221112,NaN,"[{'Aspect': '품질', 'SentimentText': '우선 마감이 좋지 ..."
93,114664,기존에 사용하던 무선이어폰이 오래되어서 배터리가 광탈하는 바람에 새로운 상품이 필요...,SNS,가전,영상/음향가전,엑스트라 TWS 블루투스 이어폰 gni-504,1,303,68,20221111,NaN,"[{'Aspect': '색상', 'SentimentText': '흰색이 훨씬 깔끔해..."


In [10]:
#원본의 df는 결측치를 제거
df.dropna(inplace=True)

In [11]:
#데이터의 개수를 1000개만 사용
df2=df[:1000]
df2['GeneralPolarity'].value_counts()

GeneralPolarity
 1.0    565
 0.0    260
-1.0    175
Name: count, dtype: int64

In [12]:
#특정 컬럼만 사용
df2=df2[['RawText', 'GeneralPolarity']]

In [13]:
#텍스트 정규화
#GeneralPolarity의 값을 int로 변경하고 0, 1, 2로 변경
def normalize(data):
	#data : df2의 인덱스 하나씩 대입 (Series)
	data['RawText']=re.sub(r'[^가-핳0-9a-zA-Z\s\.]',' ',data['RawText'])
	data['RawText']=re.sub(r'\s+',' ',data['RawText'])	

	data['GeneralPolarity']=int(data['GeneralPolarity'])
	data['GeneralPolarity'] += 1
	#data['GeneralPolarity']=data['GeneralPolarity'].map(
	#	(
	#		-1 : 0,
	#		0 : 1,
	#		1 : 2
	#	)
	#)
	return data

In [14]:
df3=df2.apply(normalize,axis=1)

In [15]:
df3.rename(
    columns={
        'GeneralPolarity' : 'labels'
    }, inplace=True
)

In [16]:
# df2.apply(lambda x : print(x), axis=1)

In [17]:
#train, test 데이터 분할
train_df, test_df=train_test_split(
    df3,test_size=0.5, random_state=42, stratify=df3['labels']
)

In [18]:
train_df['labels'].value_counts()

labels
2    283
1    130
0     87
Name: count, dtype: int64

In [19]:
#Dataset으로 생성
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))

train_ds

Dataset({
    features: ['RawText', 'labels'],
    num_rows: 500
})

In [20]:
#tokenizer 로드
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_first=False)

In [21]:
def token_fn(batch):
    result=tokenizer(
        batch['RawText'],
        truncation=True,
        max_length=128,
        padding=True,
        return_tensor='pt'
    )
    return result

train_tok=train_ds.map(token_fn, batched=True, remove_columns=['RawText'])
test_tok=test_ds.map(token_fn, batched=True, remove_columns=['RawText'])

Map: 100%|██████████| 500/500 [00:00<00:00, 1075.79 examples/s]


In [22]:
train_tok

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [23]:
#BERT + Linear 학습 모델을 선언 
class BERTCLF(nn.Module):
    def __init__(self, model_name, num_classes = 2, dropout = 0.5):
        super().__init__()
        #기존에 학습 된 모델을 로드 
        self.backbone = BertModel.from_pretrained(model_name)

        #로드한 모델의 출력 차원의 수 
        hidden = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)
        #작업의 안정성을 위해서 토크나이저의 패딩 토큰 아이디를 로드한 모델에 패딩 토큰 아이디로 사용
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
    
    def forward(self, input_ids = None, attention_mask = None, 
                labels = None, **kwargs):
        out = self.backbone(input_ids = input_ids, attnetion_mask = attention_mask)

        #CLS 부분만 추출
        pooled = out.last_hidden_state[:, 0]
        #일부 데이터 소실(과적합 방지용)
        drop_data = self.dropout(pooled)

        logits = self.fc(drop_data)

        result = {
            "Logits" : logits
        }

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            result['loss'] = loss
        
        return result

In [24]:
train_tok

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [25]:
#매개변수 **kwargs의 의미
def test_forward(input_ids=None, attention_mask=None,
labels=None, **kwargs):
    print(input_ids[0])
    print(attention_mask[0])
    print(labels)

test_forward(input_ids=train_tok[0]['input_ids'], attention_mask=train_tok[0]['attention_mask'],
             labels=train_tok[0]['labels'])

2
1
2


In [26]:
train_tok[0]
#매개변수에 * : 여러개의 인자값들을 하나의 변수에 저장
#인자에 * : 1차원 데이터의 각 원소들을 각각의 인자로 사용
#매개변수에 ** : 함수에서 생성하지 않은 매개변수가 들어올 떄 사용
#인자의 ** : 딕셔너리형 데이터를 key값을 매개변수명으로 사용하고 value를 인자로 사용하여 함수를 호출

{'labels': 2,
 'input_ids': [2,
  12208,
  4113,
  26130,
  10868,
  4017,
  8381,
  8124,
  10915,
  9082,
  9335,
  9021,
  8013,
  8534,
  18849,
  10868,
  4113,
  1664,
  8159,
  13054,
  248,
  15186,
  2005,
  2451,
  8472,
  13145,
  11514,
  2232,
  4040,
  17,
  12208,
  10868,
  4029,
  8066,
  16505,
  4042,
  347,
  11383,
  14184,
  9119,
  9747,
  4007,
  10267,
  9921,
  4061,
  8039,
  13776,
  963,
  4334,
  16505,
  4091,
  11514,
  750,
  4092,
  12208,
  4083,
  8823,
  2417,
  4158,
  17,
  3087,
  8383,
  21,
  17,
  27010,
  4008,
  3087,
  8025,
  17,
  17938,
  20277,
  4017,
  13492,
  832,
  3010,
  16707,
  8260,
  23220,
  17,
  8547,
  3089,
  2492,
  13351,
  2694,
  4128,
  1371,
  5041,
  10943,
  2009,
  8007,
  1072,
  4414,
  7968,
  10383,
  16798,
  11429,
  8262,
  4153,
  2535,
  8074,
  17,
  8108,
  1279,
  4137,
  5233,
  4017,
  9855,
  23239,
  4047,
  16612,
  4017,
  8545,
  3294,
  4598,
  4103,
  16612,
  4017,
  2483,
  10009,
  17,
  

In [27]:
test_forward(**train_tok[0])

2
1
2


In [28]:
test_dict={
    'input_ids' : [10,20,30],
    'attention_mask' : [1,1,1],
    'labels' : 2,
    'GeneralPolarity' : 1
}

test_forward(**test_dict)

10
1
2


In [29]:
test_forward(input_ids=test_dict['input_ids'], attention_mask=test_dict['attention_mask'], labels=test_dict['labels'], GeneralPolarity=test_dict['GeneralPolarity'])

10
1
2


In [30]:
#모델을 생성 (3진 분류)
model = BERTCLF(MODEL_NAME, num_classes=3)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1440.71it/s]
[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
#평가에서 사용할 함수 선언
def metrics(eval_pred):
    logits, y=eval_pred
    #logits : [x.xxxx, x.xxxx, x.xxxx]
    pred=logits.argmax(-1)

    return {
        'accuracy_score' : accuracy_score(y,pred),
        'f1_score' : f1_score(y,pred,average='macro')
    }

In [32]:
#trainer에서 사용할 설정들을 세팅 
args = TrainingArguments(
    output_dir= "/model", 
    eval_strategy= 'epoch', 
    save_strategy='epoch', 
    num_train_epochs=1, 
    learning_rate=5e-5,         #3e-05, 4e-05, 5e-05 값들을 일반적으로 사용
    weight_decay=0.01, 
    warmup_ratio= 0.1, 
    logging_steps=50, 
    load_best_model_at_end=True, 
    metric_for_best_model='f1_score', 
    greater_is_better=True,
    # dataloader_num_workers=os.cpu_count()//2      #py파일에서 사용 가능
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [33]:
trainer = Trainer(
    model = model, 
    args = args, 
    train_dataset= train_tok, 
    eval_dataset= test_tok,                 
    compute_metrics= metrics, 
    processing_class= tokenizer             # 구버전에서는 tokenizer 매개변수 
)

In [34]:
trainer.train()

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,1.016167,0.647603,0.708000,0.579093


TrainOutput(global_step=63, training_loss=0.9702680527217804, metrics={'train_runtime': 1704.452, 'train_samples_per_second': 0.293, 'train_steps_per_second': 0.037, 'total_flos': 0.0, 'train_loss': 0.9702680527217804, 'epoch': 1.0})

In [35]:
trainer.evaluate() 

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy Score,F1 Score
1.016167,0.647603,1,0.708000,0.579093


{'eval_loss': 0.6476030349731445,
 'eval_accuracy_score': 0.708,
 'eval_f1_score': 0.5790934430289106}

In [36]:
os.cpu_count() // 2     #내 컴퓨터 cpu의 코어 확인 가능

2

In [37]:
#새로운 모델을 이용해서 예측
samples=df_na['RawText'].sample(5).to_list()
samples

['식탁에서 바로 끓이고 조리고 쪄서  바로 먹을 수 있습니다. 세 식구 한 끼 준비하는데 사이즈 적당하고 색상도 튀지 않게 잔잔해서 그럭 저럭 마음에 듭니다. 막 샀을 때 새 제품 냄새가 좀 있었는데 그냥 물 한 번 끓이고 나니 괜찮아 졌지만 아무래도 음식을 하는 용기이니 만큼 냄새가 없었으면 하는 바램입니다.  밀푀유 나베 할 때 정말 잘 샀다 확신했고 당분간은 같은 메뉴가 자주 오를 것 같습니다. 제가 외출 했다 귀가가 조금 늦으면 중1짜리 아이가 만두도 쪄 먹는데 색상이나 사이즈에 의한 위화감이 덜해서 스스로 해 먹는데도 재미가 붙는 거 같습니다. 냄새때문에 별하나 뺍니다',
 '끓는 물 기능 하나 보고 그냥 바로 결정했습니다.다른 정수기 쓸 때보다 정말 이 기능 하나로 얼마나 편하게 생활이 바뀌었는지 몰라요.국 끓이거나 할 때도 시간도 덜 들고, 특히 라면 끓여먹을 때 정말로 필요합니다.냉수도 시원한데 얼음도 잘 나오고 하니 여름에 시원한 에이드 타 먹을 때 아주 편할거 같네여.냉장고에 물 받아서 얼음 얼려도 조금 손데고 덜어 먹을 때 찝찝했는데, 정수기에서 바로 얼음이 나오니 깨끗하고 좋네요.단점은 기능이 많아서 그런지 크기가 꽤 커서 공간을 많이 차지하는 점이 아쉽습니다.하지만 그건 어쩔 수 없는 거니까 만족하고 잘 쓸거 같아요.',
 '옛날 방식의 김치 냉장고를 수년 간 써오다가 허리가 아파서스탠드형 김치 냉장고로 드디어 바꾸게 되었습니다 짝짝짝 ><일단 카드 10프로 청구 할인 이라는 점이 너무 마음에 들었어요. 가격이 세다 보니깐 10프로면 몇 만원이나 할인 되는 거잖아요 ~일단 집에 와서 설치 잘했고 내부를 들여다 봤는데 330L라 그런지내부 용량은 생각보다 좁더라고요~ 그래서 많은 김치를 저장할 수 없다는게너무 아쉽습니다.ㅠㅠ 그리고 김치통 소재가 어떤 소재인지 몰라도 김치 물이너무 진하게 들어서 닦기 힘들더라고요~ 역시 장점이 있다면 단점도 있는 법이죠~SOSO한 제품입니다~',
 '학교 졸업하고 이십 년 만에 다시 재봉틀 사용해 보내요

In [38]:
tok=tokenizer(
    samples,
    padding=True,
    trucation=True,
    max_length=128,
    return_tensors='pt'
)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:2357: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [39]:
with torch.no_grad():
    out=model(**tok)
    probs=torch.softmax(out['Logits'],dim=1)

probs

tensor([[0.0693, 0.3283, 0.6024],
        [0.0295, 0.2028, 0.7678],
        [0.1899, 0.3716, 0.4385],
        [0.1622, 0.3370, 0.5007],
        [0.3057, 0.3951, 0.2993]])

In [40]:
for text, prob in zip(samples, probs):
    print(f'''
        원본 데이터 : {text},
        부정 : {prob[0] : .3f} / 중립 : {prob[1]: .3f} / 긍정 : {prob[2]: .3f}
        예측 값 : {prob.argmax()}
    ''')


        원본 데이터 : 식탁에서 바로 끓이고 조리고 쪄서  바로 먹을 수 있습니다. 세 식구 한 끼 준비하는데 사이즈 적당하고 색상도 튀지 않게 잔잔해서 그럭 저럭 마음에 듭니다. 막 샀을 때 새 제품 냄새가 좀 있었는데 그냥 물 한 번 끓이고 나니 괜찮아 졌지만 아무래도 음식을 하는 용기이니 만큼 냄새가 없었으면 하는 바램입니다.  밀푀유 나베 할 때 정말 잘 샀다 확신했고 당분간은 같은 메뉴가 자주 오를 것 같습니다. 제가 외출 했다 귀가가 조금 늦으면 중1짜리 아이가 만두도 쪄 먹는데 색상이나 사이즈에 의한 위화감이 덜해서 스스로 해 먹는데도 재미가 붙는 거 같습니다. 냄새때문에 별하나 뺍니다,
        부정 :  0.069 / 중립 :  0.328 / 긍정 :  0.602
        예측 값 : 2
    

        원본 데이터 : 끓는 물 기능 하나 보고 그냥 바로 결정했습니다.다른 정수기 쓸 때보다 정말 이 기능 하나로 얼마나 편하게 생활이 바뀌었는지 몰라요.국 끓이거나 할 때도 시간도 덜 들고, 특히 라면 끓여먹을 때 정말로 필요합니다.냉수도 시원한데 얼음도 잘 나오고 하니 여름에 시원한 에이드 타 먹을 때 아주 편할거 같네여.냉장고에 물 받아서 얼음 얼려도 조금 손데고 덜어 먹을 때 찝찝했는데, 정수기에서 바로 얼음이 나오니 깨끗하고 좋네요.단점은 기능이 많아서 그런지 크기가 꽤 커서 공간을 많이 차지하는 점이 아쉽습니다.하지만 그건 어쩔 수 없는 거니까 만족하고 잘 쓸거 같아요.,
        부정 :  0.029 / 중립 :  0.203 / 긍정 :  0.768
        예측 값 : 2
    

        원본 데이터 : 옛날 방식의 김치 냉장고를 수년 간 써오다가 허리가 아파서스탠드형 김치 냉장고로 드디어 바꾸게 되었습니다 짝짝짝 ><일단 카드 10프로 청구 할인 이라는 점이 너무 마음에 들었어요. 가격이 세다 보니깐 10프로면 몇 만원이나 할인 되는 거잖아요 ~일단 집에 와서 설치 잘했고 내부를 들여다

In [53]:
#BERT + Linear 학습 모델을 선언 
#은닉층의 CLS만 사용을 하는것이 일반적인 모델 ( transformers에 BertForSequenceClassification 객체를 이용하여 모델 )
#CLS뿐만 아니라 전체 단어 벡터의 평균, 최댓값, (CLS + mean + max)
#벡터의 평균 -> BertModel에서 결과 값( [batch_size, seq_len(max_length + 2), 768] )
    #seq_len에서 +2의 의미는  -> CLS, SEP 토큰 
class BERTCLF_Custom(nn.Module):
    def __init__(self, model_name, num_classes = 2, dropout = 0.5, kind = "cls"):
        super().__init__()
        #기존에 학습 된 모델을 로드 
        self.backbone = BertModel.from_pretrained(model_name)
        self.kind = kind

        #로드한 모델의 출력 차원의 수 
        hidden = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        #kind가 만약에 concat이라면 입력 차원의 수는 hidden * 3을 해야된다. 
        if self.kind == 'concat':
            self.fc = nn.Linear(hidden * 3, num_classes)
        else:
            self.fc = nn.Linear(hidden, num_classes)
        #작업의 안정성을 위해서 토크나이저의 패딩 토큰 아이디를 로드한 모델에 패딩 토큰 아이디로 사용
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
    
    def forward(self, input_ids = None, attention_mask = None, 
                labels = None, **kwargs):
        out = self.backbone(input_ids = input_ids, attention_mask = attention_mask)

        if self.kind == 'cls':
            #CLS 부분만 추출
            pooled = out.last_hidden_state[:, 0]
        elif self.kind == 'mean':
            #은닉층의 CLS, SEP부분을 제외하고 평균을 구한다. 
            pooled = out.last_hidden_state[:, 1:-1, :].mean(dim=1)
        elif self.kind == 'max':
            pooled = torch.max(out.last_hidden_state, dim = 1)[0]
        elif self.kind == 'concat':
            cls_data = out.last_hidden_state[:, 0, :]
            mean_data = out.last_hidden_state[:, 1 : -1, :].mean(dim=1)
            max_data = torch.max(out.last_hidden_state, dim = 1)[0]
            pooled = torch.cat(
                [cls_data, mean_data, max_data], dim=1
            )
        #일부 데이터 소실(과적합 방지용)
        drop_data = self.dropout(pooled)

        logits = self.fc(drop_data)

        result = {
            "Logits" : logits
        }

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            result['loss'] = loss
        
        return result

In [ ]:
df.iloc[1:-1,]

In [55]:
sample_data=[
    [
        [1,2,3],
        [4,5,6],
        [7,8,9]
    ],
    [
        [9,8,7],
        [6,5,4],
        [3,2,1]
    ]
]

In [56]:
sample_data = torch.tensor(sample_data, dtype = torch.float32)

In [57]:
sample_data[0][0][2]

tensor(3.)

In [58]:
mean_data = sample_data[:, 1:-1, :].mean(dim=1)

In [59]:
max_data = torch.max(sample_data, dim=1)[0]

In [60]:
cls_data = sample_data[:, 0, :]

In [61]:
#3개의 벡터를 결합
torch.cat(
    [
        cls_data, mean_data, max_data
    ], dim=1
)

tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [9., 8., 7., 6., 5., 4., 9., 8., 7.]])

In [62]:
#모델 생성
model_mean=BERTCLF_Custom(MODEL_NAME, num_classes=3, kind='mean')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1406.05it/s]
[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
trainer_mean = Trainer(
    model = model_mean, 
    args = args, 
    train_dataset= train_tok, 
    eval_dataset= test_tok, 
    compute_metrics= metrics, 
    processing_class= tokenizer             # 구버전에서는 tokenizer 매개변수 
)

trainer_mean.train()

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,0.963654,0.720500,0.676000,0.530535


TrainOutput(global_step=63, training_loss=0.9319713380601671, metrics={'train_runtime': 1314.8204, 'train_samples_per_second': 0.38, 'train_steps_per_second': 0.048, 'total_flos': 0.0, 'train_loss': 0.9319713380601671, 'epoch': 1.0})

In [64]:
#모델 생성
model_concat=BERTCLF_Custom(MODEL_NAME, num_classes=3, kind='concat')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1270.00it/s]
[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [65]:
trainer_concat=Trainer(
    model = model_concat, 
    args = args, 
    train_dataset= train_tok, 
    eval_dataset= test_tok,                 
    compute_metrics= metrics, 
    processing_class= tokenizer             # 구버전에서는 tokenizer 매개변수   
)

trainer_concat.train()

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,1.110484,0.703720,0.696000,0.517507


TrainOutput(global_step=63, training_loss=1.0721725433591813, metrics={'train_runtime': 896.9666, 'train_samples_per_second': 0.557, 'train_steps_per_second': 0.07, 'total_flos': 0.0, 'train_loss': 1.0721725433591813, 'epoch': 1.0})